In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.style.use('default')
mpl.rcParams['axes.linewidth'] = 7  # set the value globally
mpl.rcParams['xtick.major.size'] = 24
mpl.rcParams['xtick.major.width'] = 7
mpl.rcParams['xtick.minor.size'] = 16
mpl.rcParams['xtick.minor.width'] = 7
mpl.rcParams['ytick.major.size'] = 24
mpl.rcParams['ytick.major.width'] = 7
mpl.rcParams['ytick.labelsize'] = 75
mpl.rcParams['xtick.labelsize'] = 75
mpl.rcParams['ytick.minor.size'] = 16
mpl.rcParams['ytick.minor.width'] = 7
mpl.rcParams['font.size'] = 55
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['svg.fonttype'] = 'none'

In [ ]:
file ='delta_Efret_violins.csv'
save_loc = 'delta_EF.svg'
df = pd.read_csv(file)

In [ ]:

replaced_data = tmpr_df

mpl.rcParams['figure.dpi'] = 500
save_plot =True

unique_parameters = replaced_data.experimentparameter.unique()
color_palette = ['k', 'firebrick']
colors = {}

for i, param in enumerate(unique_parameters):
    colors[param] = color_palette[i]

construct = len(replaced_data.construct.unique())
parameter = len(replaced_data.experimentparameter.unique())


ylim= [ymin, ymax] = -0.12, 0.13
 

uni_construct = replaced_data.construct.unique()

#Added
func_1 = np.vectorize(lambda x : ((x-1) *20 + 1))
func_2 = np.vectorize(lambda x : ((x) * 20 + 44))

construct_array = np.array([int(x) for x in uni_construct])

pos_1 = func_1(construct_array)
pos_2 = func_2(construct_array)

xlabel_list = []
for i, pos in enumerate(pos_1):
    start = pos_1[i]
    end = pos_2[i]
    sequence = f'{start}-{end}'
    xlabel_list.append(sequence)
###

#indexer length

fig, ax = plt.subplots(1, 1, figsize=(30, 15))

tmp_df = replaced_data



row = index

y=0
con_holder = None
for x in tmp_df['plot_position']:
    current_well_n = tmp_df.loc[tmp_df['plot_position']==x, 'well_count'].iloc[0]
    ax.text(x, ymin+0.02, f'n={current_well_n}', ha='center', fontsize=24)
    
    len_param = tmp_df.experimentparameter.nunique()
    current_construct = tmp_df.loc[tmp_df['plot_position']==x, 'construct'].iloc[0]
    if con_holder is None:
        con_holder = current_construct
        comparison_positions = x #added
        y+=1
        continue
    
    if con_holder != current_construct:
        y=1
        con_holder = current_construct
        comparison_positions = x #added
        continue
    
    if y == len_param:
        y=1
        continue
        
        
    y_pos = (ymax-0.01) - y*0.0078
    x_coords = (comparison_positions, x)    #added
    
    
    star_val = tmp_df.loc[tmp_df['plot_position']==x, 'star_value']
    #ax[row].text(x, y_pos, s=star_val.iloc[0], fontsize=8, fontweight='bold', ha='center')

    ax.plot(x_coords, (y_pos, y_pos), linewidth=2, c='k') #added
    
    center_x = sum(x_coords)/len(x_coords) #added
    ax.text(center_x, y_pos+0.003, s=star_val.iloc[0], fontsize=28, fontweight='bold', ha='center') #change
    
    y+=1
ax.set_ylim(ylim)
tmp_explode = tmp_df.explode('delta_Efret_median_well')

x_positions = tmp_explode['plot_position'] + np.random.uniform(-0.05, 0.05, size=len(tmp_explode['plot_position']))
ax.scatter(x=x_positions , y=tmp_explode['delta_Efret_median_well'], s=200, c='white', edgecolor='navy', linewidth=7, zorder=2, alpha=0.4)

tmp_df['delta_Efret_median'] = tmp_df['delta_Efret_median_well'].apply(np.median)
tmp_df = tmp_df.reset_index(drop=True)
tmp_df['std_median_per_timepoint'] = tmp_df.explode(['delta_Efret_median_well']).groupby(['construct', 'experimentparameter'], as_index=False)['delta_Efret_median_well'].std().dropna()['delta_Efret_median_well'].reset_index(drop=True)

ax.scatter(x= tmp_df['plot_position'], y=tmp_df['delta_Efret_median'], s=800, c='gray', zorder=3, edgecolor='k')
ax.errorbar(x= tmp_df['plot_position'], y=tmp_df['delta_Efret_median'], yerr= tmp_df['std_median_per_timepoint'], linestyle='none', color='k',elinewidth=8, zorder=3)
tmp_explode = tmp_df.explode('delta_Efret_list_well')

iterator = 0 
 


for x in tmp_df['plot_position']:
    list_explode = tmp_explode.loc[tmp_explode['plot_position']== x]
    current_parameter = tmp_explode.loc[tmp_explode['plot_position']== x, 'experimentparameter'].iloc[0]
    
    len_dataframe = len(list_explode)
    
    len_para = tmp_explode.experimentparameter.nunique()
    if iterator == len_para:
        iterator = 0
    if 'Growth' == current_parameter:
        iterator = 0 
        
    current_color = colors[current_parameter]
    
    for sub in range(len_dataframe):
        current_dataframe = list_explode.iloc[sub]
        
        final_explode = current_dataframe['delta_Efret_list_well']
        violins =ax.violinplot(final_explode,[x], widths=0.39, showextrema=False, showmedians=False, showmeans=False)

        
        for pc in violins['bodies']:
            pc.set_facecolor('None')
            pc.set_edgecolor(current_color)
            pc.set_aplha=0.3
            pc.set_linewidth(3)

    iterator += 1

ax.set_xticks(np.arange(1, construct+1, 1))

ax.set_xticklabels(xlabel_list, fontsize='large', rotation =-45)




ax.set_ylabel('Delta Efret')
ax.set_xlabel('Linker Sequence')

patch = []
labels = ['Nontreated', 'High Infection Marker']
for color, label in zip(color_palette, labels):
    
    tmp_patch = mpatches.Patch(color=color, label=label)
    patch.append(tmp_patch)
plt.legend(title='mCherry Intensity', handles=patch, loc='upper right', bbox_to_anchor=(1.45, 1))

plt.subplots_adjust(hspace=1)
fig.suptitle('Infection: AdV5-'+'\u0394'+'E3 (Post Infection)', x=0.5, y=0.95)
if save_plot:
    plt.savefig(save_loc, format='svg', bbox_inches='tight')
plt.show()
